### **Content License Agreement**

<font color='red'><b>**WARNING**</b></font> : 본 자료는 업스테이지의 콘텐츠 자산으로, 어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다.

# 실습 — Multi-Agent Orchestration (A2A): 5가지 협업 토폴로지
**Production-Grade Agent Engineering · Colab 실습 (Upstage Solar)**

> 목표: **Agent Card·Task·Message·Part·Artifact**라는 동일한 A2A 통신 규약 위에서, **오케스트레이션 구조만 바꿔** 더 많은 케이스를 처리합니다.
> **① 순차 → ② 병렬 Fan-out/in → ③ 라우터 → ④ 피드백 루프 → ⑤ 계층형**, 그리고 무한 루프·비용·깊이 폭주를 막는 **안전장치**까지 다룹니다.

## 0. 셋업
Colab 좌측 🔑 **Secrets**에 `UPSTAGE_API_KEY`를 등록한 뒤, 아래 셀을 실행합니다.

In [1]:
!pip -q install openai

In [2]:
# Upstage Solar 셋업 — Colab 좌측 🔑 Secrets에 'UPSTAGE_API_KEY'를 등록한 뒤 실행합니다.
from google.colab import userdata
UPSTAGE_API_KEY = userdata.get("UPSTAGE_API_KEY")
from openai import OpenAI
client = OpenAI(api_key=UPSTAGE_API_KEY, base_url="https://api.upstage.ai/v1")
MODEL = "solar-pro2"

def ask(system, user, **kw):
    """Solar 단일 호출 헬퍼."""
    r = client.chat.completions.create(model=MODEL,
        messages=[{"role":"system","content":system},{"role":"user","content":user}], **kw)
    return r.choices[0].message.content.strip()
print("Solar 준비 완료:", MODEL)

TimeoutException: Requesting secret UPSTAGE_API_KEY timed out. Secrets can only be fetched when running from the Colab UI.

## 1. A2A 프리미티브 & 공용 토대
A2A의 핵심은 에이전트가 **표준 메시지**로 소통한다는 점입니다. 슬라이드의 5요소를 최소 형태로 정의합니다.
- **Agent Card**: 에이전트의 명함(이름·설명·스킬).
- **Task**: 누가(assignee)·무엇을(instruction)·어떤 맥락(context)으로.
- **Message / Part**: 결과를 이루는 조각들(텍스트·데이터).
- **Artifact**: 한 Task의 산출물(Message를 담음).

그리고 모든 토폴로지가 공유할 **안전장치 `Budget`**(스텝·깊이 상한)을 함께 정의합니다.

In [ ]:
from dataclasses import dataclass
from typing import Callable
import uuid, json, threading

# ---- A2A 5요소 (Agent Card·Task·Message·Part·Artifact) ----
@dataclass
class AgentCard:
    name: str
    description: str
    skills: list[str]

@dataclass
class Task:
    id: str
    assignee: str
    instruction: str
    context: str = ""

@dataclass
class Part:
    kind: str          # "text" | "data"
    content: object

@dataclass
class Message:
    parts: list[Part]
    def text(self) -> str:
        return "\n".join(str(p.content) for p in self.parts if p.kind == "text")

@dataclass
class Artifact:
    task_id: str
    by: str
    message: Message
    @property
    def text(self) -> str:
        return self.message.text()

# ---- 안전장치: 스텝·깊이 예산 (스레드 안전) ----
class Budget:
    def __init__(self, max_steps=8, max_depth=2):
        self.max_steps = max_steps      # 🛡 위임 횟수 상한 (비용 폭주 방지)
        self.max_depth = max_depth      # 🛡 계층 깊이 상한 (재귀 폭주 방지)
        self.steps = 0
        self._lock = threading.Lock()
    def charge(self):
        with self._lock:                # 병렬 위임에서도 안전하게 카운트
            self.steps += 1
            if self.steps > self.max_steps:
                raise RuntimeError(f"🛑 max_steps({self.max_steps}) 초과 — 비용 폭주 차단")
    def check_depth(self, depth):
        if depth + 1 > self.max_depth:
            raise RuntimeError(f"🛑 max_depth({self.max_depth}) 초과 — 계층 폭주 차단")

print("A2A 프리미티브 + Budget 정의 완료")

In [ ]:
class Specialist:
    """Agent Card + handler(task)->str. 실제 작업은 Solar(또는 도구)가 수행합니다."""
    def __init__(self, card: AgentCard, handler: Callable[[Task], str]):
        self.card = card
        self._handler = handler
    def handle(self, task: Task) -> Artifact:
        out = self._handler(task)
        return Artifact(task.id, self.card.name, Message([Part("text", out)]))

class Orchestrator:
    """직접 일하지 않고 적임자에게 Task를 위임하는 조율자. Budget을 공유해 안전하게 운영합니다."""
    def __init__(self, registry, budget=None, depth=0, name="Orchestrator"):
        self.registry = registry
        self.budget = budget or Budget()
        self.depth = depth
        self.name = name
        self.log: list = []
    def delegate(self, assignee, instruction, context="") -> Artifact:
        self.budget.charge()
        task = Task(uuid.uuid4().hex[:6], assignee, instruction, context)
        indent = "   " * (self.depth + 1)
        print(f"{indent}📨 {self.name} → {assignee}: {instruction[:34]}")
        art = self.registry[assignee].handle(task)
        self.log.append(art)
        return art

print("Specialist / Orchestrator 정의 완료")

## 2. 전문가 팀 & 멀티 토픽 뉴스
Specialist를 넉넉히 갖춰 여러 토폴로지에서 재사용합니다. 뉴스 DB도 여러 토픽으로 확장합니다.

In [ ]:
NEWS_DB = {
    "AI 에이전트": ["MCP 표준 채택 가속", "A2A 프로토콜 공개로 멀티에이전트 협업 표준화",
                  "엔터프라이즈 에이전트 도입률 급증", "에이전트 평가(Eval) 시장 부상"],
    "반도체":     ["HBM 수요 폭증으로 메모리 가격 반등", "2나노 공정 양산 경쟁 본격화",
                  "AI 가속기 자체 설계 확대", "지정학 리스크로 공급망 재편"],
    "전기차":     ["전고체 배터리 시범 양산", "충전 인프라 표준 통합 논의",
                  "보급형 모델 가격 경쟁 심화", "자율주행 SW 구독 모델 확산"],
}

# 도구형(LLM 불필요)
def research(task):
    kw = task.instruction.split(":")[-1].strip()
    hits = NEWS_DB.get(kw, [])
    return "\n".join(f"- {h}" for h in hits) or "(검색 결과 없음)"

# Solar 기반
def analyze(task):
    return ask("너는 시장 분석가입니다. 헤드라인에서 핵심 인사이트 3개를 불릿으로 뽑습니다.",
               f"[헤드라인]\n{task.context}")
def write(task):
    return ask("너는 임원 보고서 작성자입니다. 주어진 내용으로 5문장 이내의 간결한 한국어 브리핑을 씁니다.",
               f"[내용]\n{task.context}")
def synthesize(task):
    return ask("너는 편집장입니다. 여러 토픽 브리핑을 하나의 통합 임원 요약(7문장 이내)으로 종합합니다.",
               f"[토픽별 브리핑]\n{task.context}")
def summarize(task):
    return ask("너는 요약가입니다. 핵심만 3줄로 요약합니다.", task.context)
def compare(task):
    return ask("너는 비교 분석가입니다. 두 토픽의 공통점과 차이를 표로 비교합니다.", task.context)
def translate(task):
    return ask("너는 전문 번역가입니다. 주어진 한국어 텍스트를 자연스러운 영어로 번역합니다.", task.context)
def critic(task):
    return ask("너는 엄격한 편집 데스크입니다. 브리핑을 평가해 JSON만 출력합니다: "
               '{"score": 0~5 정수, "feedback": "개선 한 줄"}.', task.context)

def make_registry():
    specs = [
        Specialist(AgentCard("NewsResearcher", "키워드로 뉴스 헤드라인 수집", ["search_news"]), research),
        Specialist(AgentCard("Analyst", "헤드라인에서 인사이트 도출", ["analyze"]), analyze),
        Specialist(AgentCard("Writer", "임원용 브리핑 작성", ["write"]), write),
        Specialist(AgentCard("Synthesizer", "여러 브리핑을 하나로 종합", ["synthesize"]), synthesize),
        Specialist(AgentCard("Summarizer", "긴 글을 3줄로 요약", ["summarize"]), summarize),
        Specialist(AgentCard("Comparer", "두 토픽을 비교 분석", ["compare"]), compare),
        Specialist(AgentCard("Translator", "한국어→영어 번역", ["translate"]), translate),
        Specialist(AgentCard("Critic", "브리핑 품질 평가(0~5)", ["critique"]), critic),
    ]
    return {s.card.name: s for s in specs}

registry = make_registry()
print("등록된 전문가:", list(registry))

## 3. 토폴로지 ① 순차 파이프라인 (Sequential)
가장 기본입니다. `research → analyze → write`를 한 줄로 엮습니다. 단계가 명확하고 디버깅이 쉬운 대신, 느리고 한 단계가 막히면 전체가 멈춥니다.

In [ ]:
def run_sequential(keyword):
    orch = Orchestrator(registry, Budget(max_steps=6), name="Seq")
    head = orch.delegate("NewsResearcher", f"키워드 뉴스 검색: {keyword}").text
    ins  = orch.delegate("Analyst", "헤드라인에서 인사이트 도출", context=head).text
    rep  = orch.delegate("Writer", "인사이트로 임원 브리핑 작성", context=ins).text
    return rep, orch

report, orch = run_sequential("AI 에이전트")
print("\n===== 📄 순차 결과 =====\n" + report)
print(f"\n위임 스텝: {orch.budget.steps}")

## 4. 토폴로지 ② 병렬 Fan-out / Fan-in (Parallel)
여러 토픽을 **동시에** 처리(fan-out)하고, 결과를 **하나로 모읍니다**(fan-in). LLM 호출은 네트워크 I/O라 스레드로 실제 병렬화됩니다. `Budget`은 스레드 안전하게 공유됩니다.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def brief_one(orch, topic):
    head = orch.delegate("NewsResearcher", f"키워드 뉴스 검색: {topic}").text
    ins  = orch.delegate("Analyst", "인사이트 도출", context=head).text
    return orch.delegate("Writer", "토픽 브리핑 작성", context=ins).text

def run_parallel(topics):
    orch = Orchestrator(registry, Budget(max_steps=30), name="Par")
    # fan-out: 토픽별 브리핑을 동시에
    with ThreadPoolExecutor(max_workers=len(topics)) as ex:
        briefs = list(ex.map(lambda t: brief_one(orch, t), topics))
    combined = "\n\n".join(f"## {t}\n{b}" for t, b in zip(topics, briefs))
    # fan-in: 하나로 종합
    final = orch.delegate("Synthesizer", "토픽별 브리핑 종합", context=combined).text
    return final, orch

final, orch = run_parallel(["AI 에이전트", "반도체", "전기차"])
print("\n===== 📄 통합 요약 (fan-in) =====\n" + final)
print(f"\n위임 스텝: {orch.budget.steps} (토픽 3개 동시 처리)")

## 5. 토폴로지 ③ 라우터 (Dynamic Dispatch)
순서를 고정하지 않고, **LLM이 Agent Card를 읽어** 요청에 맞는 전문가 1명을 고릅니다. 새 전문가를 등록만 하면 라우팅 대상이 늘어납니다 — '적임자 찾기'를 말이 아니라 코드로 시연합니다.

In [ ]:
def route(request, candidates):
    cards = "\n".join(f"- {registry[c].card.name}: {registry[c].card.description}" for c in candidates)
    raw = ask("너는 라우터입니다. 요청에 가장 적합한 에이전트의 이름만 JSON으로 출력합니다: "
              '{"assignee": "<이름>"}.',
              f"[후보 에이전트]\n{cards}\n\n[요청]\n{request}")
    try:
        name = json.loads(raw[raw.find('{'): raw.rfind('}') + 1])["assignee"]
    except Exception:
        name = candidates[0]                      # 파싱 실패 시 안전 폴백
    if name not in candidates:                    # 환각 라우팅 방지
        name = candidates[0]
    return name

def run_router(request, context, candidates=("Summarizer", "Comparer", "Translator")):
    orch = Orchestrator(registry, Budget(max_steps=4), name="Router")
    assignee = route(request, list(candidates))
    print(f"   🧭 라우팅 결정: {assignee}")
    return orch.delegate(assignee, request, context=context).text, assignee

# 같은 시스템이 요청 유형에 따라 다른 전문가로 분기
sample = "\n".join(f"- {h}" for h in NEWS_DB["반도체"])
for req in ["이 내용을 3줄로 요약해줘", "이 내용을 영어로 번역해줘"]:
    out, who = run_router(req, sample)
    print(f"\n[{req}] → {who}\n{out}\n" + "-" * 40)

## 6. 토폴로지 ④ 피드백 루프 (Critic / Reflect)
초안을 **Critic**이 채점하고, 임계 미만이면 **Writer가 재작성**합니다. 품질은 오르지만 비용이 늘 수 있어 `max_rounds`로 반드시 **bounded**합니다(무한 루프 방지).

In [ ]:
def parse_verdict(raw):
    try:
        obj = json.loads(raw[raw.find('{'): raw.rfind('}') + 1])
        return int(obj.get("score", 0)), str(obj.get("feedback", ""))
    except Exception:
        return 0, raw[:80]

def run_with_critique(keyword, threshold=4, max_rounds=3):
    orch = Orchestrator(registry, Budget(max_steps=14), name="Critique")
    head = orch.delegate("NewsResearcher", f"키워드 뉴스 검색: {keyword}").text
    ins  = orch.delegate("Analyst", "인사이트 도출", context=head).text
    draft = orch.delegate("Writer", "초안 작성", context=ins).text
    for rnd in range(1, max_rounds + 1):
        score, feedback = parse_verdict(orch.delegate("Critic", "초안 평가", context=draft).text)
        print(f"   🧪 라운드 {rnd}: score={score} / {feedback[:40]}")
        if score >= threshold:
            print("   ✅ 임계 통과 — 종료")
            break
        draft = orch.delegate("Writer", f"피드백 반영 재작성: {feedback}", context=draft).text
    return draft, orch

final, orch = run_with_critique("전기차")
print("\n===== 📄 개선된 브리핑 =====\n" + final)
print(f"\n위임 스텝: {orch.budget.steps}")

## 7. 토폴로지 ⑤ 계층형 (Hierarchical / Sub-orchestrator)
**오케스트레이터도 하나의 에이전트입니다.** 상위 조율자가 `ResearchManager`(자체가 서브 오케스트레이션을 돌림)에게 통째로 위임합니다. 깊이는 `max_depth`로 제한해 재귀 폭주를 막습니다.

In [ ]:
class OrchestratorAgent:
    """오케스트레이터를 Specialist처럼 노출 → 계층 구성(A2A 합성성)."""
    def __init__(self, card, registry, budget, depth, plan):
        self.card = card
        self._registry = registry
        self._budget = budget
        self._depth = depth
        self._plan = plan
    def handle(self, task):
        self._budget.check_depth(self._depth)                 # 🛡 깊이 한도 확인
        sub = Orchestrator(self._registry, self._budget,
                           depth=self._depth + 1, name=self.card.name)
        out = self._plan(sub, task)
        return Artifact(task.id, self.card.name, Message([Part("text", out)]))

def research_team_plan(sub, task):
    topics = [t.strip() for t in task.context.split(",")]
    briefs = []
    for t in topics:
        head = sub.delegate("NewsResearcher", f"키워드 뉴스 검색: {t}").text
        ins  = sub.delegate("Analyst", "인사이트 도출", context=head).text
        briefs.append(f"## {t}\n{ins}")
    return "\n\n".join(briefs)

def run_hierarchical(topics):
    budget = Budget(max_steps=30, max_depth=2)
    reg = dict(registry)
    reg["ResearchManager"] = OrchestratorAgent(
        AgentCard("ResearchManager", "여러 토픽 리서치를 총괄(서브 오케스트레이션)", ["sub_orchestrate"]),
        reg, budget, depth=1, plan=research_team_plan)
    top = Orchestrator(reg, budget, depth=0, name="CEO")
    combined = top.delegate("ResearchManager", "토픽 리서치 총괄", context=", ".join(topics)).text
    final = top.delegate("Writer", "팀 결과로 임원 브리핑 작성", context=combined).text
    return final, budget

final, budget = run_hierarchical(["AI 에이전트", "반도체"])
print("\n===== 📄 계층형 결과 =====\n" + final)
print(f"\n총 스텝: {budget.steps} (CEO → ResearchManager → 팀)")

## 8. 안전장치 작동 확인
한도를 일부러 낮춰 가드가 실제로 막는지 봅니다. **스텝**과 **깊이** 두 가지를 모두 확인합니다.

In [ ]:
# (1) 스텝 폭주 차단 — max_steps=2로 너무 빡빡하게
tiny = Orchestrator(registry, Budget(max_steps=2), name="Tiny")
try:
    tiny.delegate("NewsResearcher", "키워드 뉴스 검색: AI 에이전트")
    tiny.delegate("Analyst", "인사이트", context="x")
    tiny.delegate("Writer", "브리핑", context="x")     # 3번째 → 차단
except RuntimeError as e:
    print("스텝 가드:", e, f"(실행 스텝 {tiny.budget.steps})")

# (2) 깊이 폭주 차단 — max_depth=1이면 ResearchManager(깊이 2)가 막힘
budget = Budget(max_steps=30, max_depth=1)
reg = dict(registry)
reg["ResearchManager"] = OrchestratorAgent(
    AgentCard("ResearchManager", "서브 오케스트레이션", ["sub_orchestrate"]),
    reg, budget, depth=1, plan=research_team_plan)
top = Orchestrator(reg, budget, depth=0, name="CEO")
try:
    top.delegate("ResearchManager", "총괄", context="AI 에이전트")
except RuntimeError as e:
    print("깊이 가드:", e)
print("→ 가드 덕분에 비용·재귀가 통제됩니다(8강 Reliability에서 회로차단·타임아웃으로 확장).")

## 직접 해보기 — 토론(Debate) 토폴로지
두 Writer가 서로 다른 관점으로 초안을 내고, **Judge**가 더 나은 쪽을 고르거나 병합하는 **토론/합의** 패턴을 추가해 보세요. 힌트: 같은 인사이트로 `write`를 관점만 바꿔 2번 위임 + Judge 1번 위임.

In [ ]:
# TODO:
# 1) 'Judge' Specialist(두 초안을 비교해 더 나은 쪽 선택/병합)를 registry에 추가
# 2) 같은 인사이트로 Writer를 관점만 바꿔 2번 위임 (예: '낙관적' vs '신중한')
# 3) 두 초안을 context로 Judge에게 위임해 최종본을 받으세요


In [ ]:
# ▼ 정답 예시 — 직접 해본 뒤 주석을 풀고 실행하세요
# def judge(task):
#     return ask("너는 편집장입니다. 두 초안 중 더 설득력 있는 쪽을 고르고, 필요하면 장점을 병합해 최종본을 만듭니다.", task.context)
# registry["Judge"] = Specialist(AgentCard("Judge", "두 초안 비교·병합", ["judge"]), judge)
# orch = Orchestrator(registry, Budget(max_steps=8), name="Debate")
# head = orch.delegate("NewsResearcher", "키워드 뉴스 검색: 반도체").text
# ins  = orch.delegate("Analyst", "인사이트", context=head).text
# a = orch.delegate("Writer", "낙관적 관점으로 브리핑", context=ins).text
# b = orch.delegate("Writer", "신중한 관점으로 브리핑", context=ins).text
# final = orch.delegate("Judge", "두 초안 비교·병합", context=f"[A]\n{a}\n\n[B]\n{b}").text
# print(final)


## 정리 — 5가지 토폴로지, 언제 쓰나
| 토폴로지 | 강점 | 주의 | 적합 상황 |
|---|---|---|---|
| ① 순차 | 단순·디버깅 쉬움 | 느림, 단일 실패점 | 단계가 분명한 선형 작업 |
| ② 병렬 Fan-out/in | 빠름(동시 처리) | 비용 동시 증가 | 독립 하위작업이 많을 때 |
| ③ 라우터 | 유연·확장 쉬움 | 라우팅 오류 위험 | 요청 유형이 다양할 때 |
| ④ 피드백 루프 | 품질↑ | 비용↑(반드시 bounded) | 품질이 중요한 산출물 |
| ⑤ 계층형 | 복잡한 목표 분할 | 깊이 폭주 위험 | 대규모·중첩 작업 |

- **불변의 핵심**: 통신 규약(Agent Card·Task·Message·Part·Artifact)은 그대로, **구조만** 바꿉니다.
- **안전장치**는 모든 구조의 전제입니다 — `max_steps`·`max_depth`로 비용·재귀를 bounded(8강 Reliability로 확장).
- 오케스트레이터도 하나의 에이전트이므로 **토폴로지는 합성됩니다**(라우터 안에 병렬, 계층 안에 피드백 …).